# 2. Submission - XGBoost

Cargar el modelo XGBoost optimizado, reentrenar sobre todo el train, predecir sobre test y enviar a Kaggle.

**Modelo:** XGBoost con hiperparámetros optimizados (RandomizedSearchCV)

**Competencia:** `playground-series-s6e2`

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
import joblib

## 2. Carga de datos

In [2]:
train = pd.read_csv("../../data/train.csv")
test = pd.read_csv("../../data/test.csv")

# Target binario
train["target"] = (train["Heart Disease"] == "Presence").astype(int)

print(f"Train: {train.shape}")
print(f"Test: {test.shape}")

Train: (630000, 16)
Test: (270000, 14)


## 3. Cargar modelo y reentrenar sobre todo el train

In [3]:
saved = joblib.load("../../models/2_xgboost_best.pkl")
best_params = saved["best_params"]
features = saved["features"]
cv_score = saved["cv_score"]

print(f"Hiperparámetros: {best_params}")
print(f"Features ({len(features)}): {features}")
print(f"ROC-AUC (CV): {cv_score:.4f}")

# Reentrenar sobre todo el dataset de train
X_train = train[features]
y_train = train["target"]

model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    **best_params
)
model.fit(X_train, y_train, verbose=False)
print(f"\nModelo entrenado sobre {len(X_train)} muestras.")

Hiperparámetros: {'colsample_bytree': np.float64(0.749816047538945), 'gamma': np.float64(0.4753571532049581), 'learning_rate': np.float64(0.22227824312530747), 'max_depth': 7, 'min_child_weight': 5, 'n_estimators': 714, 'subsample': np.float64(0.7783331011414365)}
Features (13): ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression', 'Sex', 'Chest pain type', 'FBS over 120', 'EKG results', 'Exercise angina', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
ROC-AUC (CV): 0.9519

Modelo entrenado sobre 630000 muestras.


## 4. Predicción sobre test

In [4]:
X_test = test[features]
test_probs = model.predict_proba(X_test)[:, 1]

# Submission con probabilidades (formato requerido por Kaggle)
submission = pd.DataFrame({
    "id": test["id"],
    "Heart Disease": np.round(test_probs, 4)
})

print(f"Shape: {submission.shape}")
print(f"\nEstadísticas de probabilidades:")
print(submission["Heart Disease"].describe().round(4))
print(f"\nPrimeras filas:")
submission.head(10)

Shape: (270000, 2)

Estadísticas de probabilidades:
count    270000.0000
mean          0.4497
std           0.4121
min           0.0000
25%           0.0331
50%           0.3102
75%           0.9343
max           1.0000
Name: Heart Disease, dtype: float64

Primeras filas:


,id,Heart Disease
0,630000,0.9705
1,630001,0.0092
2,630002,0.9791
3,630003,0.0095
4,630004,0.0619
5,630005,0.9838
6,630006,0.0039
7,630007,0.4043
8,630008,0.9903
9,630009,0.0140


## 5. Guardar CSV

In [5]:
SUBMISSION_FILE = "2_XGB_submission.csv"
submission.to_csv(SUBMISSION_FILE, index=False)

# Verificar
check = pd.read_csv(SUBMISSION_FILE)
print(f"Archivo: {SUBMISSION_FILE}")
print(f"Shape: {check.shape}")
print(f"Columnas: {list(check.columns)}")
print(f"IDs: {check['id'].min()} - {check['id'].max()}")
check.head()

Archivo: 2_XGB_submission.csv
Shape: (270000, 2)
Columnas: ['id', 'Heart Disease']
IDs: 630000 - 899999


,id,Heart Disease
0,630000,0.9705
1,630001,0.0092
2,630002,0.9791
3,630003,0.0095
4,630004,0.0619


## 6. Submit a Kaggle

In [6]:
COMPETITION = "playground-series-s6e2"
MESSAGE = "XGBoost tuned - 13 features - RandomizedSearchCV"

!kaggle competitions submit -c {COMPETITION} -f {SUBMISSION_FILE} -m "{MESSAGE}"

Successfully submitted to Predicting Heart Disease



  0%|          | 0.00/3.83M [00:00<?, ?B/s]
  0%|          | 16.0k/3.83M [00:00<01:09, 57.5kB/s]
  5%|▍         | 192k/3.83M [00:00<00:06, 614kB/s]  
 14%|█▍        | 544k/3.83M [00:00<00:03, 1.10MB/s]
 80%|███████▉  | 3.05M/3.83M [00:00<00:00, 7.17MB/s]
100%|██████████| 3.83M/3.83M [00:02<00:00, 1.91MB/s]
